In [1]:
# =========================================
# 3.3 Data Construction — create new features / modeling table
# Inputs:
#   prep_outputs/final_selected_table_imputed.csv  (clean+imputed, target non-missing)
# Outputs:
#   prep_outputs/final_modeling_table.csv
#   prep_outputs/construction_summary_table.csv
#   figs/*.png  (diagnostic plots)
# =========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------
# Paths
# ------------------------------
os.makedirs("figs", exist_ok=True)
os.makedirs("prep_outputs", exist_ok=True)

SRC = "prep_outputs/final_selected_table_imputed.csv"
OUT = "prep_outputs/final_modeling_table.csv"
SUM = "prep_outputs/construction_summary_table.csv"

df = pd.read_csv(SRC)

RID_COL = "RID"
TARGET_COL = "AMIGR"

# ------------------------------
# Track shape before construction
# ------------------------------
shape_before = df.shape
cols_before = df.shape[1]

# ------------------------------
# 1) Demographic transformations
# ------------------------------
# Age bands
if "AGE_P" in df.columns:
    df["AGE_BAND"] = pd.cut(
        pd.to_numeric(df["AGE_P"], errors="coerce"),
        bins=[17, 29, 44, 59, np.inf],
        labels=["18-29", "30-44", "45-59", "60+"]
    )

# BMI categories
if "BMI" in df.columns:
    df["BMI_CAT"] = pd.cut(
        pd.to_numeric(df["BMI"], errors="coerce"),
        bins=[0, 18.5, 25, 30, np.inf],
        labels=["Underweight", "Normal", "Overweight", "Obese"]
    )

# ------------------------------
# 2) Behavioral / lifestyle
# ------------------------------
# Alcohol intensity = days / period_units (guard against 0)
if {"ALC12MYR", "ALC12MTP"}.issubset(df.columns):
    days = pd.to_numeric(df["ALC12MYR"], errors="coerce")
    units = pd.to_numeric(df["ALC12MTP"], errors="coerce").replace(0, np.nan)
    df["ALC_INTENSITY"] = (days / units).fillna(0)

# Work exposure buckets
if "YRSWRKPA" in df.columns:
    df["WORK_EXP_CAT"] = pd.cut(
        pd.to_numeric(df["YRSWRKPA"], errors="coerce"),
        bins=[-1, 5, 15, 25, np.inf],
        labels=["0-5", "6-15", "16-25", "26+"]
    )

# ------------------------------
# 3) Core composites (NEW specs)
# ------------------------------
# 3.1 PAIN_INDEX = pain frequency × pain intensity
# PAIN_2A (frequency): 1 Never, 2 Some days, 3 Most days, 4 Every day
# Map to 0/1/2/3 so "Never" contributes 0.
# PAIN_4 (intensity): 1 A little, 3 Somewhere between, 2 A lot -> map 1/2/3
pain_freq_map = {1: 0, 2: 1, 3: 2, 4: 3}
pain_int_map  = {1: 1, 3: 2, 2: 3}  # increasing with intensity

if {"PAIN_2A", "PAIN_4"}.issubset(df.columns):
    pf = pd.to_numeric(df["PAIN_2A"], errors="coerce").map(pain_freq_map)
    pi = pd.to_numeric(df["PAIN_4"],  errors="coerce").map(pain_int_map)
    df["PAIN_INDEX"] = (pf * pi).astype("float")
    # Plot
    df["PAIN_INDEX"].plot(kind="hist", bins=20)
    plt.title("Pain Index (frequency × intensity)")
    plt.xlabel("Index (0–9)")
    plt.tight_layout()
    plt.savefig("figs/fig_3_3_pain_index.png", dpi=200)
    plt.close()

# 3.2 MENTAL_HEALTH_SCORE (Iteration2 style: reverse then sum; higher=worse)
# Items: ASISAD, ASINERV, ASIRSTLS, ASIHOPLS, ASIEFFRT, ASIWTHLS
mdi_items_all = ["ASISAD","ASINERV","ASIRSTLS","ASIHOPLS","ASIEFFRT","ASIWTHLS"]
mdi_items = [c for c in mdi_items_all if c in df.columns]

def reverse_minmax(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    vals = s.dropna().values
    if vals.size == 0:
        return s
    lo, hi = np.nanmin(vals), np.nanmax(vals)
    return (lo + hi) - s

if mdi_items:
    mdi_rev = pd.DataFrame({c: reverse_minmax(df[c]) for c in mdi_items})
    df["MENTAL_HEALTH_SCORE"] = mdi_rev.sum(axis=1, min_count=1)  # sum of reversed items
    # Plot
    df["MENTAL_HEALTH_SCORE"].plot(kind="hist", bins=25)
    plt.title("Mental Health Score (reversed-item sum)")
    plt.xlabel("Higher = more distress")
    plt.tight_layout()
    plt.savefig("figs/fig_3_3_mental_health.png", dpi=200)
    plt.close()

# 3.3 Sleep sufficiency (helper for EDA & interactions)
if "ASISLEEP" in df.columns:
    hours = pd.to_numeric(df["ASISLEEP"], errors="coerce")
    df["SLEEP_SUFFICIENT"] = pd.Series(
        np.where(hours.between(7, 9, inclusive="both"), 1, 0),
        index=df.index
    ).astype("Int64")
    df["SLEEP_SUFF_LABEL"] = df["SLEEP_SUFFICIENT"].map({1: "Sufficient", 0: "Insufficient"}).astype("string")
    # Plot
    df["SLEEP_SUFF_LABEL"].value_counts(dropna=False).reindex(["Sufficient","Insufficient"]).plot(kind="bar", rot=0)
    plt.title("Sleep sufficiency (derived from ASISLEEP)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("figs/fig_3_3_sleep_sufficient.png", dpi=200)
    plt.close()

# ------------------------------
# 4) Interaction terms
# ------------------------------
if {"ASISLEEP", "DOINGLWA"}.issubset(df.columns):
    df["SLEEP_X_WORK"] = pd.to_numeric(df["ASISLEEP"], errors="coerce") * pd.to_numeric(df["DOINGLWA"], errors="coerce")

if {"SEX", "ALCSTAT"}.issubset(df.columns):
    df["SEX_X_ALC"] = pd.to_numeric(df["SEX"], errors="coerce") * pd.to_numeric(df["ALCSTAT"], errors="coerce")

# ------------------------------
# 5) Verification & summary
# ------------------------------
shape_after = df.shape
cols_after = df.shape[1]

print("Dataset shape before construction:", shape_before)
print("Dataset shape after construction :", shape_after)
print(f"New features added: {cols_after - cols_before}")
print("New columns (if present):", [c for c in [
    "AGE_BAND","BMI_CAT","ALC_INTENSITY","WORK_EXP_CAT",
    "PAIN_INDEX","MENTAL_HEALTH_SCORE","SLEEP_SUFFICIENT","SLEEP_X_WORK","SEX_X_ALC"
] if c in df.columns])

summary_constructed = pd.DataFrame({
    "Remaining NaNs": df.isna().sum(),
    "Unique Values": df.nunique(),
    "Data Type": df.dtypes
})
summary_constructed.to_csv(SUM)
print("Saved construction summary table ->", SUM)

# ------------------------------
# 6) Save final modeling table
# ------------------------------
df.to_csv(OUT, index=False)
print("Saved final modeling table ->", OUT)
print("Final dataset shape (saved):", df.shape)

# ------------------------------
# 7) Additional plots for the report
# ------------------------------
# Age band distribution
if "AGE_BAND" in df.columns:
    df["AGE_BAND"].value_counts().sort_index().plot(kind="bar")
    plt.title("Age band distribution")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("figs/fig_3_3_age_band.png", dpi=200)
    plt.close()

# BMI category distribution
if "BMI_CAT" in df.columns:
    df["BMI_CAT"].value_counts().sort_index().plot(kind="bar")
    plt.title("BMI category distribution")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("figs/fig_3_3_bmi_cat.png", dpi=200)
    plt.close()


Dataset shape before construction: (25403, 51)
Dataset shape after construction : (25403, 59)
New features added: 8
New columns (if present): ['AGE_BAND', 'BMI_CAT', 'WORK_EXP_CAT', 'PAIN_INDEX', 'MENTAL_HEALTH_SCORE', 'SLEEP_SUFFICIENT', 'SLEEP_X_WORK']
Saved construction summary table -> prep_outputs/construction_summary_table.csv
Saved final modeling table -> prep_outputs/final_modeling_table.csv
Final dataset shape (saved): (25403, 59)
